# Add MACD Features to CSV
This notebook reads `system/data/raw/data.csv`, computes MACD features, and writes the result to `system/data/feature/data.csv`.

In [1]:
# Imports and path setup
from pathlib import Path
import sys
sys.path.insert(0, '/workspaces/quant')
import pandas as pd
from system.src.features.macd_divergence import *

# Resolve source and destination paths
abs_src = Path('/workspaces/quant/system/data/raw/data.csv')
rel_src = Path.cwd() / 'system' / 'data' / 'raw' / 'data.csv'
src_path = abs_src if abs_src.exists() else rel_src
if not src_path.exists():
    raise FileNotFoundError(f'Raw data not found at {abs_src} or {rel_src}')

dest_path = Path('/workspaces/quant/system/data/feature/data.csv')
dest_path.parent.mkdir(parents=True, exist_ok=True)
src_path, dest_path

(PosixPath('/workspaces/quant/system/data/raw/data.csv'),
 PosixPath('/workspaces/quant/system/data/feature/data.csv'))

In [2]:
# Load raw data
df = pd.read_csv(src_path, parse_dates=['time'])
df.head(2)

,time,open,high,low,close,volume,symbol
0,2010-01-04,1.47,1.49,1.47,1.49,603120,HPG
1,2010-01-05,1.56,1.56,1.49,1.56,1203080,HPG


In [ ]:
from system.src.features.macd_divergence import *
df_feat = calculate_macd(df)

NameError: name 'calculate_macd' is not defined

In [ ]:

# Compute all MACD-based features
# 1) MACD + signal + histogram (also leaves EMA columns)
df_feat = calculate_macd(df)
# 2) Histogram extrema flags
df_feat = find_hist_extrema(df_feat)
# 3) Divergence signals and references
df_feat = find_macd_divergence(df_feat)

# Preview columns
sorted(df_feat.columns.to_list())

In [3]:
# Write to feature CSV and verify
df_feat.to_csv(dest_path, index=False)
print(f'Wrote features to: {dest_path}')

# Show a concise preview of feature columns
cols = [
    'time','symbol','close',
    'ema_close_26','ema_close_12',
    'macd','macd_signal','macd_hist',
    'hist_peak_strict','hist_trough_strict','hist_peak_local','hist_trough_local',
    'div_bullish','div_bearish','div_bullish_t1','div_bearish_p1'
]
print('Columns present:', [c for c in cols if c in df_feat.columns])
df_feat.tail(5)[[c for c in cols if c in df_feat.columns]]

Wrote features to: /workspaces/quant/system/data/feature/data.csv
Columns present: ['time', 'symbol', 'close', 'ema_close_26', 'ema_close_12', 'macd', 'macd_signal', 'macd_hist', 'hist_peak_strict', 'hist_trough_strict', 'hist_peak_local', 'hist_trough_local', 'div_bullish', 'div_bearish', 'div_bullish_t1', 'div_bearish_p1']


,time,symbol,close,ema_close_26,ema_close_12,macd,macd_signal,macd_hist,hist_peak_strict,hist_trough_strict,hist_peak_local,hist_trough_local,div_bullish,div_bearish,div_bullish_t1,div_bearish_p1
3915,2025-09-15,HPG,30.35,28.057993,28.906203,0.848209,0.684889,0.163320,False,False,True,False,False,False,NaT,NaT
3916,2025-09-16,HPG,29.95,28.198142,29.066787,0.868645,0.721640,0.147005,False,False,False,False,False,False,NaT,NaT
3917,2025-09-17,HPG,29.20,28.272354,29.087281,0.814927,0.740297,0.074630,False,False,False,False,False,False,NaT,NaT
3918,2025-09-18,HPG,29.15,28.337365,29.096930,0.759566,0.744151,0.015415,False,False,False,False,False,False,NaT,NaT
3919,2025-09-19,HPG,29.25,28.404967,29.120479,0.715512,0.738423,-0.022911,False,False,False,False,False,False,NaT,NaT
